<a href="https://colab.research.google.com/github/ak7396/tmdb-eda-assignment/blob/main/tmdb_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import requests

API_KEY = userdata.get("API_KEY")

url = "https://api.themoviedb.org/3/discover/movie"

params = {
    "api_key": API_KEY,
    "language": "en-US",
    "sort_by": "popularity.desc",
    "page": 1
}

response = requests.get(url, params=params)

print(response.status_code)
data = response.json()
movies = data.get("results", [])

print("Movies fetched:", len(movies))

200
Movies fetched: 20


CONVERING INTO DATFRAME

In [ ]:

import pandas as pd
df = pd.DataFrame(movies)

# Select useful columns (clean dataset)
df = df[[
    "id", "title", "release_date", "popularity",
    "vote_average", "vote_count", "genre_ids", "overview"
]]

print(df.shape)

(20, 8)


In [ ]:
import sqlite3

# Convert problematic column
df["genre_ids"] = df["genre_ids"].astype(str)

conn = sqlite3.connect("tmdb_movies.db")

df.to_sql("movies", conn, if_exists="replace", index=False)

conn.commit()

# Verify
df_check = pd.read_sql("SELECT * FROM movies", conn)

print(df_check.shape)
print(df_check.head())

conn.close()

(20, 8)
        id                       title release_date  popularity  vote_average  \
0  1523145   Your Heart Will Be Broken   2026-03-26   1165.1266         6.900   
1    83533        Avatar: Fire and Ash   2025-12-17    503.6662         7.300   
2  1115544  Mike & Nick & Nick & Alice   2026-03-14    352.9397         6.700   
3  1084187               Pretty Lethal   2026-03-13    311.0726         6.745   
4  1297842                        GOAT   2026-02-11    297.8194         7.700   

   vote_count        genre_ids  \
0          24      [10749, 18]   
1        2074    [878, 12, 14]   
2         109    [35, 878, 80]   
3         157  [28, 53, 10402]   
4         249  [16, 35, 10751]   

                                            overview  
0  High school student Polina is saved from bully...  
1  In the wake of the devastating war against the...  
2  Two gangsters and the woman they love try to s...  
3  A troupe of ballerinas find themselves fightin...  
4  A small goat with big 

Load from SQLite

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("tmdb_movies.db")

df = pd.read_sql("SELECT * FROM movies", conn)

conn.close()

In [ ]:
df.head()

,id,title,release_date,popularity,vote_average,vote_count,genre_ids,overview
0,1523145,Your Heart Will Be Broken,2026-03-26,1165.1266,6.900,24,"[10749, 18]",High school student Polina is saved from bully...
1,83533,Avatar: Fire and Ash,2025-12-17,503.6662,7.300,2074,"[878, 12, 14]",In the wake of the devastating war against the...
2,1115544,Mike & Nick & Nick & Alice,2026-03-14,352.9397,6.700,109,"[35, 878, 80]",Two gangsters and the woman they love try to s...
3,1084187,Pretty Lethal,2026-03-13,311.0726,6.745,157,"[28, 53, 10402]",A troupe of ballerinas find themselves fightin...
4,1297842,GOAT,2026-02-11,297.8194,7.700,249,"[16, 35, 10751]",A small goat with big dreams gets a once-in-a-...


In [ ]:
df.describe()

,id,popularity,vote_average,vote_count
count,2.000000e+01,20.000000,20.000000,20.000000
mean,1.062134e+06,263.099260,6.932400,587.650000
std,3.097655e+05,234.540901,0.806402,656.930444
min,8.353300e+04,101.340400,5.200000,2.000000
25%,8.789158e+05,151.561075,6.482750,145.000000
50%,1.137552e+06,181.292600,6.950000,399.500000
75%,1.271912e+06,300.460725,7.381750,761.500000
max,1.523145e+06,1165.126600,8.500000,2408.000000


In [ ]:
df.isnull().sum()

,0
id,0
title,0
release_date,0
popularity,0
vote_average,0
vote_count,0
genre_ids,0
overview,0


In [ ]:
df["genre_ids"].value_counts()

,count
genre_ids,
"[10749, 18]",2
"[878, 12, 14]",1
"[35, 878, 80]",1
"[28, 53, 10402]",1
"[16, 35, 10751]",1
"[28, 80, 53]",1
"[878, 12]",1
"[27, 9648, 80]",1
"[27, 53, 35]",1


In [ ]:
df["genre_ids"] = df["genre_ids"].str.replace("[", "", regex=False)
df["genre_ids"] = df["genre_ids"].str.replace("]", "", regex=False)
df["genre_ids"] = df["genre_ids"].str.split(",")

df_exploded = df.explode("genre_ids")

df_exploded["genre_ids"] = df_exploded["genre_ids"].str.strip()

df_exploded["genre_ids"].value_counts()

,count
genre_ids,
35,6
878,6
53,6
12,5
80,5
18,5
27,4
10751,4
16,3
